In [9]:
import pandas as pd
import folium
import re
from pathlib import Path
from branca.colormap import linear
from folium.plugins import HeatMap
import numpy as np

# charger
p = Path("../data/raw/ski-resorts.csv")
df = pd.read_csv(p, encoding="utf-8", engine="python")

# parse coord (conserver ta fonction existante si elle est déjà dans le notebook)
def parse_coord(s):
    if pd.isna(s): return (None, None)
    s = str(s).strip()
    m = re.search(r"POINT\s*\(?\s*([-\d\.]+)[ ,]+([-\d\.]+)\s*\)?", s, re.IGNORECASE)
    if m:
        return float(m.group(2)), float(m.group(1))
    mlat = re.search(r"lat['\"]?\s*[:=]\s*['\"]?(-?\d+(?:\.\d+)?)", s, re.IGNORECASE)
    mlon = re.search(r"(?:lon|long|lng)['\"]?\s*[:=]\s*['\"]?(-?\d+(?:\.\d+)?)", s, re.IGNORECASE)
    if mlat and mlon:
        return float(mlat.group(1)), float(mlon.group(1))
    nums = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", s)
    if len(nums) >= 2:
        a, b = float(nums[0]), float(nums[1])
        if abs(a) <= 90 and abs(b) <= 180: return a, b
        if abs(b) <= 90 and abs(a) <= 180: return b, a
    return (None, None)

df[['lat','lon']] = df['location_coordinate'].apply(lambda x: pd.Series(parse_coord(x)))

df_coords = df.dropna(subset=['lat','lon']).copy()
if df_coords.empty:
    raise ValueError("Aucune coordonnée valide trouvée.")

# détecter colonne elevation
elev_col = next((c for c in df_coords.columns if 'elev' in c.lower() or 'elevation' in c.lower()), None)
if elev_col:
    df_coords[elev_col] = pd.to_numeric(df_coords[elev_col], errors='coerce')

# détecter colonne "hauteur de neige" (candidats: snow, depth, snowfall)
snow_col = next((c for c in df_coords.columns if any(k in c.lower() for k in ('snow','depth','snowfall'))), None)
if snow_col:
    df_coords[snow_col] = pd.to_numeric(df_coords[snow_col], errors='coerce')

# carte centrée
center = [df_coords['lat'].astype(float).mean(), df_coords['lon'].astype(float).mean()]
m = folium.Map(location=center, zoom_start=6)

# --- INVERSION: couleur = elevation, taille = snowfall ---

# colormap pour l'élévation (si dispo)
if elev_col and df_coords[elev_col].notna().any():
    emin = float(df_coords[elev_col].min())
    emax = float(df_coords[elev_col].max())
    cmap = linear.YlOrRd_09.scale(emin, emax)
    cmap.caption = f"{elev_col}"
    cmap.add_to(m)
else:
    cmap = None

# taille par snowfall -> radius range (si dispo)
min_radius, max_radius = 3, 14
if snow_col and df_coords[snow_col].notna().any():
    smin = float(df_coords[snow_col].min())
    smax = float(df_coords[snow_col].max())
    if smax == smin:
        df_coords['_radius'] = (min_radius + max_radius) / 2
    else:
        df_coords['_radius'] = df_coords[snow_col].astype(float).clip(smin, smax).apply(
            lambda v: min_radius + (v - smin) / (smax - smin) * (max_radius - min_radius)
        )
else:
    df_coords['_radius'] = (min_radius + max_radius) / 2

# ajouter points (sans clustering)
pts = folium.FeatureGroup(name="Stations")
for _, r in df_coords.iterrows():
    lat, lon = float(r['lat']), float(r['lon'])
    radius = float(r['_radius'])
    # couleur basée sur elevation
    if cmap is not None and pd.notna(r.get(elev_col)):
        color = cmap(float(r[elev_col]))
    else:
        color = "#444444"
    popup = folium.Popup(
        f"{r.get('name','')}<br>{r.get('location_country','')}<br>{elev_col}: {r.get(elev_col,'N/A')}<br>{snow_col}: {r.get(snow_col,'N/A')}",
        max_width=300
    )
    folium.CircleMarker(
        location=(lat, lon),
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=popup
    ).add_to(pts)
pts.add_to(m)

# légende HTML pour expliquer taille -> snowfall et couleur -> elevation
emin_text = f"{emin:.0f}" if 'emin' in locals() else "N/A"
emax_text = f"{emax:.0f}" if 'emax' in locals() else "N/A"
smin_text = f"{smin:.1f}" if 'smin' in locals() else "N/A"
smax_text = f"{smax:.1f}" if 'smax' in locals() else "N/A"

legend_html = f"""
<div style="position: fixed; bottom: 50px; left: 10px; z-index:9999; background:white; padding:8px; border-radius:4px;
            box-shadow:0 0 6px rgba(0,0,0,0.3); font-size:12px;">
  <b>Couleur = {elev_col}</b><br>
  min: {emin_text} &nbsp; max: {emax_text}<br>
  <b>Taille des points = {snow_col}</b><br>
  min: {smin_text} &nbsp; max: {smax_text}<br>
  radius ~ {min_radius}..{max_radius} px
</div>"""
from branca.element import Element
m.get_root().html.add_child(Element(legend_html))

folium.LayerControl(collapsed=False).add_to(m)
out = "ski_resorts_points_map.html"
m.save(out)
print("Carte enregistrée sous", out)

Carte enregistrée sous ski_resorts_points_map.html
